In [ ]:
!git lfs install
!git clone https://huggingface.co/Qwen/Qwen3-4B-Base

Git LFS initialized.


In [3]:
%pwd

'c:\\Post Training'

In [4]:
!git clone https://huggingface.co/Qwen/Qwen3-4B-Base

Cloning into 'Qwen3-4B-Base'...
Filtering content:  33% (1/3)
Filtering content:  66% (2/3)
Filtering content:  66% (2/3), 3.77 GiB | 1.45 MiB/s
Filtering content: 100% (3/3), 3.77 GiB | 1.45 MiB/s
Filtering content: 100% (3/3), 7.49 GiB | 1.32 MiB/s
Filtering content: 100% (3/3), 7.49 GiB | 2.80 MiB/s, done.


In [3]:
import json, torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
p = "Qwen3-4B-Base"

for f in sorted(os.listdir(p)):
    fp = os.path.join(p, f)
    if os.path.isfile(fp):
        print(f"{f:<40} {os.path.getsize(fp)}")


.gitattributes                           1554
LICENSE                                  11544
README.md                                2993
config.json                              757
generation_config.json                   144
merges.txt                               1823241
model-00001-of-00003.safetensors         3957900840
model-00002-of-00003.safetensors         3987450520
model-00003-of-00003.safetensors         99630640
model.safetensors.index.json             33224
tokenizer.json                           7334926
tokenizer_config.json                    9916
vocab.json                               2776833


In [4]:
cuda = torch.cuda.is_available()
dtype = torch.float16 if cuda else torch.float32 
token = AutoTokenizer.from_pretrained("Qwen3-4B-Base")
model = AutoModelForCausalLM.from_pretrained("Qwen3-4B-Base", torch_dtype = dtype).eval()
if cuda:
    model.cuda()
token.pad_token = token.pad_token or token.eos_token
print("Qurilma:", model.device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qurilma: cuda:0


In [5]:
SAVOLLAR = [
    "Python nima?",
    "Menga samarali o‘qish uchun 3 ta maslahat ber.",
    "Bugun havo issiq — shu gapni ingliz tiliga tarjima qil.",
    "2 + 2 * 3 nechaga teng?",
]

In [6]:
inp = token("Python nima?", return_tensors="pt").to(model.device)

In [7]:
inp

{'input_ids': tensor([[30280,   308,  7523,    30]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1]], device='cuda:0')}

In [18]:
out = model.generate(**inp, max_new_tokens = 100, do_sample = False,
                     repetition_penalty = 1.1, pad_token_id = token.pad_token_id)
out

tensor([[30280,   308,  7523,    30, 13027,  2804,   437,   352,   374,    71,
          6651,  8579,  5267,   334, 30280,   334,  1959,  1031, 20178,   294,
           559,  1085,   988, 11956,  2780,   379, 38191,   330, 91626,  4128,
             1,   320,    67,   559,  1085,   988, 10266,     8,  4222, 13515,
           518,   278,   329, 10881,    11,   259,   449,   318,   742, 11164,
          1463,   360,  1268,  5205,   276,   294,   559,  1085,   988, 10266,
           307,   404,    13,   547,   220,    16,    24,    23,    15, 12034,
           483,  3177, 18952, 20679, 11164,   220,    16,    24,    24,    15,
         12034, 66259,    64, 12157,    78,  5242, 16465,  1242,   311,  1645,
         55204,   379, 46857,   321, 29428,   708,     6,  2740,    11,  9876,
         81490,   595, 26892, 49016]], device='cuda:0')

In [19]:
token.decode(out[0][inp.input_ids.shape[-1]:], skip_special_tokens=True).strip()

'Python qanday ishlaydi?\n**Python** — bu universal dasturlash tillari yoki "programming language" (dasturlash til) deb ham ataladigan, tizimli va modulierlangan dasturlash tilidir. U 1980-yillarning oxiri va 1990-yillarda Guido van Rossum tomonidan yaratilgan bo\'lib, bugungi kunda dun'

In [10]:
def javob(savol):
    input = token(savol, return_tensors = "pt").to(model.device)
    out = model.generate(**input, max_new_tokens = 100, do_sample = False, repetition_penalty = 1.1,
    pad_token_id = token.pad_token_id)
    return token.decode(out[0], skip_special_tokens = True).strip()

In [11]:
torch.cuda.is_available()

True

In [12]:
javoblar = [{"Savol":s, "base_javob": javob(s)} for s in SAVOLLAR]

In [13]:
javoblar

[{'Savol': 'Python nima?',
  'base_javob': 'Python nima? Python qanday ishlaydi?\n**Python** — bu universal dasturlash tillari yoki "programming language" (dasturlash til) deb ham ataladigan, tizimli va modulierlangan dasturlash tilidir. U 1980-yillarning oxiri va 1990-yillarda Guido van Rossum tomonidan yaratilgan bo\'lib, bugungi kunda dun'},
 {'Savol': 'Menga samarali o‘qish uchun 3 ta maslahat ber.',
  'base_javob': "Menga samarali o‘qish uchun 3 ta maslahat ber. Menga samarali o'qish uchun quyidagi 3 ta maslahatni qo'llang:\n\n1. **O'zgaruvchanlik va tajribalash**: O'zingizga mos bo'lgan usullarni toping va unlar bilan ishlashingiz mumkin. Masalan, bir necha kitoblarga e'tibor bering yoki elektronik platformalar orqali dars"},
 {'Savol': 'Bugun havo issiq — shu gapni ingliz tiliga tarjima qil.',
  'base_javob': 'Bugun havo issiq — shu gapni ingliz tiliga tarjima qil. Bu so\'zlar to\'g\'ri yozilsa, ularning ma\'nosi nima bo\'ladi? "Bugun havo issiq" so\'zlari ingliz tilida quyidagi

In [15]:
json.dump(javoblar, open("javoblar.json", "w", encoding = "utf-8"), 
          ensure_ascii=False, indent=2)

In [18]:
from datasets import load_dataset

ds = load_dataset('yahma/alpaca-cleaned', split='train')
ds = ds.shuffle(seed = 42).select(range(1500))


In [19]:
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 1500
})

In [20]:
def to_prompt_completion(ex):
    q = ex["instruction"]

    if ex["input"]:
        q += "n\n" + ex["input"]
    
    return {
            "prompt":[{"role": 'user', "content": q}],
            "completion": [{"role": "assistant", "content": ex["output"]}]
            }


In [21]:
ds = ds.map(to_prompt_completion, remove_columns=ds.column_names)
ds = ds.train_test_split(test_size=150, seed = 42)

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [22]:
ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1350
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 150
    })
})

In [23]:
from trl import SFTConfig, SFTTrainer

In [24]:
cfg = SFTConfig(
    output_dir='my_llm_sft_v1',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=10,
    learning_rate=1e-5,
    lr_scheduler_type= 'cosine',
    num_train_epochs=3,
    warmup_ratio=0.2,
    bf16=True,
    gradient_checkpointing=True,
    max_length=2048,
    completion_only_loss=True,
    eval_strategy="steps",
    eval_steps= 50
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [25]:
trainer = SFTTrainer(model=model, args=cfg, processing_class = token, 
                        train_dataset = ds['train'], eval_dataset = ds['test'])

Tokenizing train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

print(torch.cuda.max_memory_allocated() / 1e9, "GB")   # 4-dars uchun yozib qo‘ying!
 
trainer.save_model("my-llm-v1-sft")
token.save_pretrained("my-llm-v1-sft")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
